In [ ]:
import os
import subprocess
import chemiscope
import ipi
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read
import nqetools as nqe
from ase.visualize import view
# This follows:
# https://atomistic-cookbook.org/examples/pi-metad/pi-metad.html

In [ ]:
# Make a directory to store everything
directory_md = "md"
directory_metamd = "data"#"metamd"
directory_metapimd = "metapimd"
n_beads = 8
timestep = 1.0
total_steps = 5000
stride = 10
temperature = 298

In [ ]:
atoms = read("h5o2+.xyz", index=-1)
atoms.center(vacuum=20.0)

In [ ]:
view(atoms)

In [ ]:
# Run unbiased MD
# Make sure the directory is empty
nqe.remove_directory(directory_md)
# Run the calculation
nqe.run_md(directory_md, atoms,
           driver='zundel',
           md_type="NVT-GLE",
           # xml_in="input-md.xml",
           n_beads=1,
           timestep=timestep,
           total_steps=total_steps,
           stride=stride,
           temperature=temperature,
           )

In [ ]:
# Run metadynamics
# Make sure the directory is empty
nqe.remove_directory(directory_metamd)
# Run the calculation
atoms = nqe.run_plumed_md(directory_metamd, atoms,
                          driver='zundel',
                          md_type="NVT-GLE",
                          n_beads=1,
                          timestep=timestep,
                          total_steps=total_steps,
                          stride=stride,
                          temperature=temperature,
                          plumed_type="mtd-coord",
                          )

In [ ]:
output_data, output_desc = ipi.read_output(os.path.join(directory_metamd, "md.out"))
colvar_data = ipi.read_trajectory(os.path.join(directory_metamd,"md.colvar_0"), format="extras")[
    "d,c1.lessthan,c2.lessthan,dc,mtd.bias"
]
traj_data = ipi.read_trajectory(os.path.join(directory_metamd, "md.pos_0.xyz"))

In [ ]:
nqe.plot_time_potential_bias(output_data)

In [ ]:
nqe.plot_time_temperature(output_data)

In [ ]:

sum_hills_str= "plumed sum_hills --hills HILLS --min 0.21,-1 --max 0.31,1 --bin 100,100 --outfile FES --stride 100 --mintozero"


with open(os.path.join(directory_metamd,"plumed.dat"), "r") as file:
    subprocess.run(sum_hills_str.split(), stdin=file,text=True,
    )

